# Collection des données de Booking.com avec Beautifulsoup et Selenium

In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd
import time

# On récupère la liste des villes où l'on veut chercher des hôtels
df_meteo_complet = pd.read_csv("src/top_destination.csv")
# On nettoie un peu le tableau pour enlever la colonne d'index inutile
df_meteo_complet.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')


hotels_liste = []

# selenium va permettre d'utiliser un vrai navigateur pour simuler une navigation, évitant ainsi certains blocage anti-bot récents de booking
options = webdriver.FirefoxOptions()
driver = webdriver.Firefox(options=options)

print("--- DÉBUT DU SCRAPING ---")

#Pour chaque ville
for index, row in df_meteo_complet.iterrows():
    print(f"Chargement de Booking pour la ville de : {row['ville']}...")

    # On demande au navigateur d'aller sur la page de résultats Booking
    url = f"https://www.booking.com/searchresults.fr.html?ss={row['ville']}"
    driver.get(url)

    # On attend 6 secondes pour que la page charge bien le contenu
    time.sleep(6)

    # On récupère tout le code HTML de la page actuelle
    html_source = driver.page_source
    soup = BeautifulSoup(html_source, "html.parser")

    # On repère les blocs qui contiennent les cartes des hôtels
    hotel_cards = soup.find_all("div", {"data-testid": "property-card"})

    # On prend seulement les 20 premiers hôtels de la page
    for card in hotel_cards[:20]:

        # On cherche le nom de l'hôtel
        title_box = card.find("div", {"data-testid": "title"})
        nom_hotel = title_box.text.strip() if title_box else "Non renseigné"

        # On cherche le lien vers la page de l'hôtel
        link_box = card.find("a", {"data-testid": "title-link"})
        url_hotel = link_box["href"] if link_box else "Non renseigné"

        # On cherche la note donnée par les clients
            ### ->>  On a pas réussi à trouver une approche générique au niveau des class booking change ses css régulièrement pour bloquer les outil de scrapping
        score_box = card.find("div", class_="f63b14ab7a dff2e52086")
        if not score_box:
            score_box = card.find("div", class_="f63b14ab7a")

        # On transforme le texte de la note en valeur numérique avec un '.'
        note_finale = None
        if score_box:
            note_texte = score_box.text.strip()
            note_texte = note_texte.replace('[', '').replace(']', '').replace(',', '.')
            try:
                note_finale = float(note_texte)
            except ValueError:
                note_finale = None

        # On ajoute toutes ces infos dans notre liste principale (les données GPS sont déjà inclusent dans le fichier source de ce notebook)
        hotels_liste.append({
            "id": row["id"],
            "nom_hotel": nom_hotel,
            "url_hotel": url_hotel,
            "note": note_finale
        })

#fermeture du navigateur
driver.quit()

# Formatage (en vue de l'export)
df_hotels = pd.DataFrame(hotels_liste)

print(f"--- SCRAPING RÉUSSI : {len(df_hotels)} hôtels enregistrés ! ---")

g:\prog_data_scientist\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


--- DÉBUT DU SCRAPING AVEC SELENIUM (WINDOWS) & BEAUTIFULSOUP ---
Chargement de la page Booking pour : Mont Saint Michel...
Chargement de la page Booking pour : St Malo...
Chargement de la page Booking pour : Bayeux...
Chargement de la page Booking pour : Le Havre...
Chargement de la page Booking pour : Rouen...
Chargement de la page Booking pour : Paris...
Chargement de la page Booking pour : Amiens...
Chargement de la page Booking pour : Lille...
Chargement de la page Booking pour : Strasbourg...
Chargement de la page Booking pour : Chateau du Haut Koenigsbourg...
Chargement de la page Booking pour : Colmar...
Chargement de la page Booking pour : Eguisheim...
Chargement de la page Booking pour : Besancon...
Chargement de la page Booking pour : Dijon...
Chargement de la page Booking pour : Annecy...
Chargement de la page Booking pour : Grenoble...
Chargement de la page Booking pour : Lyon...
Chargement de la page Booking pour : Gorges du Verdon...
Chargement de la page Booking pour : 

In [2]:
# test résultat
df_hotels.head(20)

,id,nom_hotel,url_hotel,note
0,1,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,8.2
1,1,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,8.4
2,1,Hotel De La Digue,https://www.booking.com/hotel/fr/de-la-digue.f...,7.6
3,1,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,8.3
4,1,La Confiance,https://www.booking.com/hotel/fr/les-terrasses...,7.5
5,1,Le Duguesclin,https://www.booking.com/hotel/fr/le-duguesclin...,7.9
6,1,Hotel Gabriel,https://www.booking.com/hotel/fr/hotel-gabriel...,8.2
7,1,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,8.4
8,1,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,7.5
9,1,La Mère Poulard,https://www.booking.com/hotel/fr/la-mere-poula...,7.6


### merge de la partie api et de la partie booking

In [3]:
df_ville_meteo_booking = pd.merge(df_meteo_complet,df_hotels,how='inner',on='id')

In [ ]:
# export pour le Aws bucket S3
df_ville_meteo_booking.to_csv("src/kayak_ds.csv", index=False)